In [2]:
import os
import numpy as np
import time
from pathlib import Path
import pickle
import pandas as pd
import traceback
from scipy import stats
import matplotlib.pyplot as plt
from galaxy_ellipse_collection import GalaxyEllipseCollection

#load configs
import sys
from config import db_connection, sys_path, results_output_directory, pickle_file
os.environ['TANGOS_DB_CONNECTION'] = '/home/bk639/data_base/CDM_all_shapes.db'
os.environ['TANGOS_PROPERTY_MODULES'] = 'mytangosproperty'
sys.path.append(sys_path)
import tangos

with open(pickle_file, 'rb') as f:
    ellipse_dict = pickle.load(f)


/home/bk639/miniconda3/envs/pynbody/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from MCMC_sims import get_keys,smooth_shape

In [ ]:
def plot_combined_shapes(ellipse_dict, verbose=False):
    for sim in ellipse_dict:
        for halo in ellipse_dict[sim]:
            try:
                h = tangos.get_halo(halo)
                hid  = h.basename.split('_')[1]
                reff = get_keys(sim,hid, (0,0),'Reff')
                a = h['a_s']
                b = h['b_s']
                c = h['c_s']
                rbins = h['rbins_s']
                reff = get_keys(sim, hid, (0.0, 0.0), 'Reff')

                rbins_f, a_f, b_f, c_f, a_s_func, b_s_func, c_s_func = smooth_shape(rbins, a, b, c, k=3)
                # get a,b,c at 2*reff
                # a_s = a_s_func(2 * Reff)
                # b_s = b_s_func(2 * Reff)
                # c_s = c_s_func(2 * Reff)

                ba_s = lambda r: b_s_func(r)/a_s_func(r)
                ca_s = lambda r: c_s_func(r)/a_s_func(r)
                star_rbins = rbins_f
                ba_s_raw = b_f/a_f
                ca_s_raw = c_f/a_f
                dm_rbins = []


                # reff = h['image_reffs_v'][0]
                # ba_s = h.calculate('ba_s_smoothed()')
                # ca_s = h.calculate('ca_s_smoothed()')
                # star_rbins = h.calculate('rbins_s')
                # ba_d = h.calculate('ba_d_smoothed()')
                # ca_d = h.calculate('ca_d_smoothed()')
                # dm_rbins = h.calculate('rbins_d')
                # # raw (unsmoothed) data
                # ba_s_raw = h['ba_s']
                # ca_s_raw = h['ca_s']
                # ba_d_raw = h['ba_d']
                # ca_d_raw = h['ca_d']
            except Exception as e:
                print(f'Error loading data for {sim} halo {halo}: {e}')
                continue

            try:
                if len(star_rbins) > 0:
                    f, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), sharex=True, dpi=300)
                    plt.subplots_adjust(hspace=0.05)
                    colors = {'DM': ['blue', 'darkblue'], 'Stars': ['red', 'darkred']}
                    s = 50

                    # ---- Stars ----
                    r_smooth = np.linspace(min(star_rbins), max(max(star_rbins), 2 * reff), 100)
                    ax1.plot(r_smooth, ba_s(r_smooth), c=colors['Stars'][0], linestyle='-', lw=3, label='Stars')
                    ax2.plot(r_smooth, ca_s(r_smooth), c=colors['Stars'][0], linestyle='-', lw=3)
                    # raw scatter
                    ax1.scatter(star_rbins, ba_s_raw, c=colors['Stars'][1], s=s)
                    ax2.scatter(star_rbins, ca_s_raw, c=colors['Stars'][1], s=s)

                    if max(star_rbins) < 2 * reff:
                        print(f'{sim} {halo} max(rbins) < 2reff')
                        print(f'{sim} {halo} ca_s at 2reff: {ca_s(2 * reff)}')
                if len(dm_rbins) > 0:
                    # ---- Dark matter ----
                    r_smooth_d = np.linspace(min(dm_rbins), max(dm_rbins), 100)
                    ax1.plot(r_smooth_d, ba_d(r_smooth_d), c=colors['DM'][0], linestyle='-', lw=3, label='DM')
                    ax2.plot(r_smooth_d, ca_d(r_smooth_d), c=colors['DM'][0], linestyle='-', lw=3)
                    # raw scatter
                    ax1.scatter(dm_rbins, ba_d_raw, c=colors['DM'][1], s=s)
                    ax2.scatter(dm_rbins, ca_d_raw, c=colors['DM'][1], s=s)

                for ax in (ax1, ax2):
                    for j in range(2, 4):
                        ax.axvline(j * reff, c='k', alpha=0.5, lw=2, ls='--')
                    ax.axvline(0.5, c='gray', alpha=1, lw=3, ls='-.', label='0.5 kpc')
                    ax.axvline(reff, c='k', alpha=1, label=r'R$_{eff}$', lw=3, ls='--')
                    ax2.set_xlabel('R [kpc]', fontsize=30)
                    ax.set_ylim([0, 1])
                    ax.tick_params(which='both', labelsize=20)
                    ax.grid(False)

                ax1.legend(loc='lower right', prop={'size': 20})
                ax1.set_ylabel('Q = B/A', fontsize=30)
                ax2.set_ylabel('S = C/A', fontsize=30)
                ax1.set_xlim([0, 3.1 * reff])

                if not os.path.exists('3DShapes'):
                    os.makedirs('3DShapes')
                filename = f'3DShapes/{sim}.Shapes.{hid}.png'
                f.savefig(filename, bbox_inches='tight', pad_inches=.1)
                plt.close(f)

            except Exception as e:
                print(traceback.format_exc())
                print(f"An error processing sim {sim} halo {halo}: {e}")

# Usage
plot_combined_shapes(ellipse_dict)

/home/bk639/miniconda3/envs/pynbody/lib/python3.14/site-packages/scipy/interpolate/_fitpack_py.py:307: RuntimeWarning: The maximal number of iterations (20) allowed for finding smoothing
spline with fp=s has been reached. Probable cause: s too small.
(abs(fp-s)/s>0.001)
  res = _impl.splrep(x, y, w, xb, xe, k, task, s, t, full_output, per, quiet)


r634.romulus25.3072g1HsbBH r634.romulus25.3072g1HsbBH/%/halo_1 max(rbins) < 2reff
r634.romulus25.3072g1HsbBH r634.romulus25.3072g1HsbBH/%/halo_1 ca_s at 2reff: 0.18966740098766083
rogue.cosmo25cmb.4096g5HbwK1BH rogue.cosmo25cmb.4096g5HbwK1BH/%/halo_22 max(rbins) < 2reff
rogue.cosmo25cmb.4096g5HbwK1BH rogue.cosmo25cmb.4096g5HbwK1BH/%/halo_22 ca_s at 2reff: 0.42331594133179523


In [ ]:
faceon_average_reffs = []

for sim in ellipse_dict:
    for halo in ellipse_dict[sim]:
        h = tangos.get_halo(halo)
        try:
            reffs = h['image_reffs_v']
            faceon_average_reffs.append(np.average(reffs)/reffs[0])
        except Exception as e:
            print(f'Error loading reffs for {sim} halo {halo}: {e}')
        break
    break

print(np.average(faceon_average_reffs))





In [5]:
faceon_average_reffs


[]

In [6]:
h['rbins_s']

SimArray([ 0.15939342,  0.22197312,  0.26544935,  0.30150394,  0.33270885,
           0.36198563,  0.38885803,  0.41416964,  0.43839926,  0.46163659,
           0.4842354 ,  0.50622125,  0.52745124,  0.54840374,  0.56943644,
           0.5900523 ,  0.61022484,  0.63047243,  0.65032845,  0.66994845,
           0.68954485,  0.70898538,  0.72816463,  0.74727571,  0.76635787,
           0.78544732,  0.80444446,  0.82371205,  0.84327575,  0.86253053,
           0.88199152,  0.90166392,  0.92103293,  0.94039023,  0.95990116,
           0.97954248,  0.99914634,  1.01897684,  1.03855766,  1.05853036,
           1.07860205,  1.09876172,  1.1190534 ,  1.13928544,  1.15968877,
           1.17961441,  1.20002584,  1.22027381,  1.24072747,  1.26149845,
           1.28188564,  1.30256551,  1.32296953,  1.34331812,  1.36408   ,
           1.38498376,  1.40611219,  1.42667246,  1.44643168,  1.46535055,
           1.48307176,  1.50206156,  1.52259633,  1.54360165,  1.56565649,
           1.58764171,  1

In [7]:
string = 'KF_low'
print('KF' in string)

True


In [8]:
print(r"M$_* < 10^8 \mthrm{M}_{\odot}$")

M$_* < 10^8 \mthrm{M}_{\odot}$


In [6]:
plt.figure()


<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

In [ ]:
plt.title(r"M$_* < 10^8 \text{M}_{\odot}$")